# Decoder Transformer — Mini-GPT (char-level)

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Um Transformer decoder-only (GPT) é o mesmo bloco de encoder, mas toda camada de atenção é *causal*: a posição $t$ não pode olhar para posições $> t$. Treinamos para prever o próximo token e amostramos autoregressivamente na inferência.


## Formulação Matemática

Máscara causal $M$ com $M_{ij} = 1$ se $j \leq i$, caso contrário $0$:

$$\text{CausalAttn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + (1 - M)\cdot (-\infty)\right) V$$

A loss é a cross-entropy do próximo token:

$$\mathcal{L} = -\sum_t \log p_\theta(x_{t+1} \mid x_{\leq t}).$$


## Implementação


In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_len, max_len)).bool()
        self.register_buffer('mask', mask)
    def forward(self, x):
        B, N, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = (t.view(B, N, self.n_heads, self.d_head).transpose(1, 2) for t in (q, k, v))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        scores = scores.masked_fill(~self.mask[:N, :N], float('-inf'))
        attn = self.drop(F.softmax(scores, dim=-1))
        ctx = (attn @ v).transpose(1, 2).contiguous().view(B, N, D)
        return self.out(ctx)

class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab, d_model=128, n_heads=4, d_ff=512, n_layers=2, max_len=128):
        super().__init__()
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads, d_ff, max_len) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.head.weight = self.tok.weight  # weight tying
    def forward(self, ids):
        pos = torch.arange(ids.size(1), device=ids.device)
        x = self.tok(ids) + self.pos(pos)
        for b in self.blocks:
            x = b(x)
        return self.head(self.ln(x))


## Experimento


In [ ]:
# Toy corpus: just a string of "abcabcabc..."
text = ("abcdefg " * 200).strip()
vocab = sorted(set(text))
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])

def get_batch(block=32, batch=16):
    ix = torch.randint(0, len(data) - block - 1, (batch,))
    x = torch.stack([data[i:i+block] for i in ix])
    y = torch.stack([data[i+1:i+1+block] for i in ix])
    return x, y

model = MiniGPT(vocab=len(vocab))
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

for step in range(300):
    x, y = get_batch()
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, len(vocab)), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f'step {step:4d}  loss {loss.item():.3f}')


In [ ]:
@torch.no_grad()
def sample(model, prompt, n=40):
    ids = torch.tensor([stoi[c] for c in prompt])[None]
    for _ in range(n):
        logits = model(ids[:, -64:])[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, nxt], dim=1)
    return ''.join(itos[i.item()] for i in ids[0])

print(sample(model, 'abcd'))


## Discussão

- A máscara é a *única* diferença estrutural entre encoder e decoder.
- Compartilhar pesos (weight tying) entre o embedding de entrada e a projeção de saída economiza parâmetros e melhora a qualidade.
- Para treino de qualidade real, troque o texto de brinquedo por linguagem natural tokenizada e treine por muito mais passos.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
